In [ ]:
!pip install transformers accelerate bitsandbytes flask pyngrok

# Step 1: Loading the Large Language Model (LLM)

In this section, we initialize the **Qwen-2.5-Coder-3B-Instruct** model from Hugging Face. To ensure the model runs efficiently within the available GPU memory (VRAM) constraints, the following optimizations are implemented:

* **4-Bit Quantization:** Utilizing `BitsAndBytesConfig` to compress the model weights, significantly reducing the memory footprint without substantial loss in reasoning capabilities.
* **Flash Attention 2:** An optimized attention mechanism that reduces memory usage and increases processing speed, especially important for handling the extensive `prompt_template.txt`.
* **Device Mapping:** The model is automatically mapped to the GPU (`cuda`) to ensure fast inference times for SPARQL generation.


In [ ]:
import torch
from transformers import pipeline , BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model_id = "Qwen/Qwen2.5-Coder-3B-Instruct"

pipe = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"quantization_config": quantization_config},
    device_map="auto",

    max_length=None
)

print("Qwen-Coder is ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Qwen-Coder is ready


In [ ]:
from google.colab import userdata
from pyngrok import ngrok

try:
  NGROK_TOKEN = userdata.get('NGROK_TOKEN')
  ngrok.set_auth_token(NGROK_TOKEN)

  public_url = ngrok.connect(5000)
  print('tunnel is successfuly connected')
  print(f'public URL: {public_url}')

except Exception as e:
  print('something goes wrong')

tunnel is successfuly connected
public URL: NgrokTunnel: "https://kolby-unpondered-sniffingly.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/ask", methods=["POST"])
def ask_model():
  data = request.json

  instruction = data.get("instruction", "")

  question = data.get("question", "")

  full_input = f"{instruction}\n\n**Research Question:** {question}"


  messages = [
      {"role" : "user" , "content" : full_input}
  ]

  outputs = pipe(messages, max_new_tokens=1024, do_sample=False, pad_token_id=pipe.tokenizer.eos_token_id, max_length=None)
  response_text = outputs[0]["generated_text"][-1]["content"]

  return jsonify({
        "sparql_query": response_text
    })


if __name__ == "__main__":
    app.run(port=5000)


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
INFO:werkzeug:127.0.0.1 - - [08/Mar/2026 19:12:27] "POST /ask HTTP/1.1" 200 -
Both `max_new_tokens` (=1024) and `max

In [ ]:
from google.colab import runtime
runtime.unassign()